# Lecture 3.3 demo: the Farm tracker

ASM 532, Lecture 3.3. The code from the slides, in the same order, as cells to run top to bottom:

1. **The Farm tracker:** the object model, Crow's Foot, and the data model
2. **From data model to SQL:** building a table, joins, and what the database will and won't enforce
3. **Objects in Python:** the `farm.py` read-along: classes, inheritance, encapsulation and polymorphism

Lines marked *farm.py* are the 2024 script's own notes.

## The Farm tracker: object model

**Task domain:** tracking the locations of my crops and animals

| Object | Attributes | Operations | Related to |
| --- | --- | --- | --- |
| **Farm** | name | add, view | has 1-many Fields. has 0-many Animals. |
| **Field** | name, acres, field_type (pasture, cropland), note | add, view | has 0-many plots. If pasture, has 0-many Animals. |
| **Plot** | name, length, width, planted_date, harvest_date | add, view, plant, harvest | has 0-1 crop |
| **Crop** | name, days_to_maturity | add, view | - |
| **Animal** | name, species, breed, birthdate, weight | calculate age, details | - |
| **Goat** | secret name | same as Animal | is a type of Animal |
| **Cow** | - | details of its own | is a type of Animal |

Section 2 builds these as SQL tables. Section 3 builds the **Animal**, **Goat** and **Cow** rows as Python classes.

## Crow's Foot legend

![The four Crow's Foot combinations. A bar then a ring: minimum zero, maximum one, optional. A ring then a fork: minimum zero, maximum many, optional. Two bars: minimum one, maximum one, mandatory. A bar then a fork: minimum one, maximum many, mandatory.](../img/m3-creately-crows-foot-cardinality-3.webp)

Source: [Crow's Foot Notation guide](https://www.creately.com/guides/crows-foot-notation/), Creately.

## The Farm tracker: data model

The same model as tables, in Crow's Foot notation. (The diagram draws in JupyterLab and Notebook 7. Other editors may show the text that describes it instead.) `real` is SQL's name for a decimal number, a `float` in Python.

```mermaid
erDiagram
    accTitle: Farm tracker, full data model
    accDescr: Seven tables. Farm has a name. Field has a name, acres, a note, a farm_id foreign key and a field_type_id foreign key to the field type lookup table. Plot has a name, length, width, planted and harvest dates, and foreign keys field_id and an optional crop_id. Crop has a name and days to maturity. Animal has a name, breed, birthdate, weight and a goat-only secret name, with foreign keys farm_id, an optional field_id, and species_id to the species lookup table. One farm has one or more fields and zero or more animals; one field has zero or more plots and, if a pasture, zero or more animals; one crop is grown in zero or more plots, and a plot has zero or one crop.
    FARM ||--|{ FIELD : has
    FARM ||--o{ ANIMAL : has
    FIELD ||--o{ PLOT : has
    FIELD |o--o{ ANIMAL : pastures
    CROP |o--o{ PLOT : "grown in"
    FIELD_TYPE ||--o{ FIELD : classifies
    SPECIES ||--o{ ANIMAL : classifies
    FARM {
        int farm_id PK
        string name
    }
    FIELD {
        int field_id PK
        int farm_id FK
        int field_type_id FK
        string name
        real acres
        string note
    }
    FIELD_TYPE {
        int field_type_id PK
        string name "pasture, cropland"
    }
    PLOT {
        int plot_id PK
        int field_id FK
        int crop_id FK "optional"
        string name
        int length
        int width
        date planted_date
        date harvest_date
    }
    CROP {
        int crop_id PK
        string name
        int days_to_maturity
    }
    ANIMAL {
        int animal_id PK
        int farm_id FK
        int field_id FK "optional"
        int species_id FK
        string name
        string breed
        date birthdate
        real weight
        string secret_name "goats only"
    }
    SPECIES {
        int species_id PK
        string name "goat, cow"
    }
```

## From data model to SQL

The SQL from the lecture slides, in the same order:

1. building a table
2. the farm data, so far
3. joins: the foreign keys at work
4. inserts the database accepts, but shouldn't
5. joins that catch them

This uses Python's built-in `sqlite3` with an in-memory database, so nothing is written to disk. Run the cells in order.

In [ ]:
import sqlite3

db = sqlite3.connect(":memory:")
db.execute("PRAGMA foreign_keys = ON;")

def run(sql):
    """Run one or more SQL statements."""
    db.executescript(sql)

def show(query):
    """Run a SELECT and print its rows under their column names."""
    cur = db.execute(query)
    print(" | ".join(col[0] for col in cur.description))
    for row in cur.fetchall():
        print(" | ".join("NULL" if v is None else str(v) for v in row))

### 1. Building a table

The slides show `animal`; the other six tables follow the same pattern.

- `not null` = required. If a column is not `not null`, it is optional: Animal's `field_id`, and Plot's `crop_id` (a plot can sit empty).
- A lookup table (`field_type`, `species`) is a real table with its own rows. SQLite has no `enum` type.

In [ ]:
run("""
create table farm (
    farm_id integer primary key,
    name text not null
);

create table field_type (
    field_type_id integer primary key,
    name text not null
);
insert into field_type (field_type_id, name) values (1, 'pasture'), (2, 'cropland');

create table field (
    field_id integer primary key,
    farm_id integer not null,
    field_type_id integer not null,
    name text not null,
    acres real,
    note text,
    foreign key (farm_id) references farm (farm_id),
    foreign key (field_type_id) references field_type (field_type_id)
);

create table crop (
    crop_id integer primary key,
    name text not null,
    days_to_maturity integer
);

create table plot (
    plot_id integer primary key,
    field_id integer not null,
    crop_id integer,                -- optional: a plot can sit empty
    name text,
    length integer,
    width integer,
    planted_date text,
    harvest_date text,
    foreign key (field_id) references field (field_id),
    foreign key (crop_id) references crop (crop_id)
);

create table species (
    species_id integer primary key,
    name text not null
);
insert into species (species_id, name) values (1, 'goat'), (2, 'cow');

create table animal (
    animal_id integer primary key,
    farm_id integer not null,
    field_id integer,               -- optional: only pasture animals have a field
    species_id integer not null,
    name text not null,
    breed text,
    birthdate text,
    weight real,
    secret_name text,               -- goats only
    foreign key (farm_id) references farm (farm_id),
    foreign key (field_id) references field (field_id),
    foreign key (species_id) references species (species_id)
);
""")

### 2. The farm data, so far

Two farms, three fields, two crops, three plots (one empty), and the two animals from `farm.py`. Sunny the goat is in the pasture; Bessy the cow is in the barn, with no field.

In [ ]:
run("""
insert into farm (farm_id, name) values (1, 'Home farm'), (2, 'River farm');

insert into field (field_id, farm_id, field_type_id, name, acres) values
    (1, 1, 1, 'North pasture', 12.5),
    (2, 1, 2, 'South field', 8.0),
    (3, 2, 2, 'River field', 20.0);

insert into crop (crop_id, name, days_to_maturity) values (1, 'carrot', 70), (2, 'lettuce', 45);

insert into plot (plot_id, field_id, crop_id, name, planted_date) values
    (1, 2, 1, 'south plot A', '2026-05-15'),
    (2, 2, NULL, 'south plot B', NULL),
    (3, 3, 2, 'river plot', '2026-06-01');

insert into animal (animal_id, farm_id, field_id, species_id, name, breed, birthdate, weight, secret_name) values
    (1, 1, 1, 1, 'Sunny', 'highland', '2009-04-01', 20, 'secret goaty goat'),
    (2, 1, NULL, 2, 'Bessy', 'Holstein', '2010-04-01', 200, NULL);
""")
for table in ["farm", "field", "crop", "plot", "animal"]:
    print(table)
    show(f"select * from {table}")
    print()

### 3. Joins: the foreign keys at work

Which crops are growing in which fields? A **join** lines up rows from two tables where a key matches: `plot.field_id = field.field_id`. `join` drops rows with no match; `left join` keeps them, so the empty plot stays, with its crop as `NULL`. Many crops, many fields: through Plot.

In [ ]:
show("""
select field.name as field, plot.name as plot, crop.name as crop
from plot
join field on plot.field_id = field.field_id
left join crop on plot.crop_id = crop.crop_id
order by field.name, plot.name
""")

### 4. Inserts the database accepts, but shouldn't

Our model has two rules: animals only go in pastures, and an animal's farm must match its field's farm. Does the database enforce them?

In [ ]:
run("""
-- Home farm, South field (cropland), goat
insert into animal (farm_id, field_id, species_id, name) values (1, 2, 1, 'Clover');
-- Home farm, River field (River farm's), goat
insert into animal (farm_id, field_id, species_id, name) values (1, 3, 1, 'Pepper');
""")
show("select animal_id, name, farm_id, field_id from animal")

Both accepted. Compare a rule the database **does** enforce: a foreign key that points at nothing. There is no species 99.

In [ ]:
try:
    run("insert into animal (farm_id, species_id, name) values (1, 99, 'Mystery');")
except sqlite3.IntegrityError as error:
    print("Refused:", error)

### 5. Joins that catch them

Which animals are on a different farm from their field? Clover's farm is right (her field is the wrong *type*), and Bessy has no field, so the `join` skips her. Join one table further, to `field_type`, to catch animals outside a pasture. This is the checking your code has to do.

In [ ]:
print("Animals whose farm doesn't match their field's farm:")
show("""
select animal.name, animal.farm_id, field.farm_id as field_farm_id
from animal
join field on animal.field_id = field.field_id
where animal.farm_id != field.farm_id
""")

print()
print("Animals in a field that isn't a pasture:")
show("""
select animal.name, field.name as field, field_type.name as field_type
from animal
join field on animal.field_id = field.field_id
join field_type on field.field_type_id = field_type.field_type_id
where field_type.name != 'pasture'
""")

# Objects in Python

The `farm.py` read-along, one cell per slide.

## Let's try to create a more reusable animal

*farm.py:* look for help using `help(date)`. Indentation matters. Note the use of the colon. Note the use of `self`. Order within the function doesn't matter; order when passing into the function matters, hence documentation matters.

In [10]:
from datetime import date

class Animal:
    def __init__(self, name, birth_year, birth_month, birth_day, weight):
        self.name = name
        self.birthdate = date(birth_year, birth_month, birth_day)
        self.weight = weight

## Demo using the animal

In [11]:
bessy = Animal("Bessy", 2010, 4, 1, 200)
print(f"hello {bessy.name}")

hello Bessy


## Example stub class

In [12]:
class Dodo(Animal):
    pass

## Goat as an example of inheritance

*farm.py:* create a child class.

In [13]:
class Goat(Animal):
    def __init__(self, name, birth_year, birth_month, birth_day, weight, breed):
        super().__init__(name, birth_year, birth_month, birth_day, weight)

        self.breed = breed
        goat_secret_name = "secret goaty goat"
        self.goat_secret_name = "secret goaty goat"

## Later: come write a method

*farm.py:* provide a calculated output without revealing the details of this object. Then: print the details.

This cell re-defines `Animal`, now with two methods.

In [14]:
class Animal:
    def __init__(self, name, birth_year, birth_month, birth_day, weight):
        self.name = name
        self.birthdate = date(birth_year, birth_month, birth_day)
        self.weight = weight

    def calculate_age(self):
        today = date.today()
        age = today.year - self.birthdate.year - ((today.month, today.day) < (self.birthdate.month, self.birthdate.day))
        return age

    def details(self):
        print(f"{self.name} is {self.calculate_age()} years old today!")

`Goat` was built from the *old* `Animal`, which had no methods, so it doesn't have them yet. Run the `Goat` class again (this is the same code as above) so it inherits from the new one.

In [15]:
class Goat(Animal):
    def __init__(self, name, birth_year, birth_month, birth_day, weight, breed):
        super().__init__(name, birth_year, birth_month, birth_day, weight)

        self.breed = breed
        goat_secret_name = "secret goaty goat"
        self.goat_secret_name = "secret goaty goat"

## Demo goat object

*farm.py:* access goat secret name and explain why that's bad.

In [16]:
sunny = Goat("Sunny", 2009, 4, 1, 20, "highland")
sunny.details()
print(sunny.goat_secret_name)

Sunny is 17 years old today!
secret goaty goat


## Cow is an example of polymorphism

*farm.py:* overriding the previously defined function.

In [17]:
class Cow(Animal):
    def __init__(self, name, birth_year, birth_month, birth_day, weight, breed):
        super().__init__(name, birth_year, birth_month, birth_day, weight)
        self.breed = breed

    def details(self):
        print(f"{self.name} is {self.calculate_age()} years old today. She is a {self.breed} weighing {self.weight} pounds.")

In [18]:
bessy = Cow("Bessy", 2010, 4, 1, 200, "Holstein")
bessy.details()

Bessy is 16 years old today. She is a Holstein weighing 200 pounds.
